# 04 - Image channel switched off on ViProKG (1 GPU, about 0.1-0.2 GPU-hours)

Completes the "off" column of the image ablation for ViProKG at the signed configuration (MENTOR_base+L,
lambda_align 8, dim 400, lr 5e-4, g = 0.15): **3 runs with `--use_image 0`** (seeds 42, 7, 2024), everything else
identical to the signed run. The matching real-image runs are not re-run: their per-key ranks are shipped in the
release (`data/experimental/v3/ranks_v2/anh_v2_real_s*`) and reproduce the signed ranks bit for bit (5,274 of 5,274 keys
at every seed). The existing on/off control of ViProKG was measured at lambda_align 0, which is why it is not reused.

This round was added after the signed scope of the permutation experiment (notebook 03); its criterion below was written
before the run.

**Add Data (2 datasets, internet may stay OFF):** (1) the release dataset (v1.2.0 or later: it must contain
`notebooks/`, `scripts/score_anh_v2.py` with `--tag_b` and `data/experimental/v3/ranks_v2/anh_v2_real_s42`);
(2) the same PhoBERT checkpoint dataset (`vinai/phobert-base`, path containing "phobert") used for notebook 03.

## Pre-specified criterion (written before any run cell executes)

**|delta overall (real - off)| <= 0.65 MRR points** (3-seed mean, test, both directions): the same neutrality criterion
as the W/Y control (B7).

* PASS -> the image channel is neutral on ViProKG at lambda 8.
* FAIL -> switching images on or off changes the result on ViProKG at lambda 8.

Reported without thresholds: delta by group (overall / ordinary / slice) x (all / mention-overlap / mention-free) with the
95% across-seed CI (t, df = 2) and the query-level paired bootstrap CI (B = 1000).

In [1]:
print("""PRE-SPECIFIED (written before the run cells):
  |delta overall (real - off)| <= 0.65  ->  image channel neutral on ViProKG at lambda 8
  group deltas: reported, with CIs, no thresholds""")

PRE-SPECIFIED (written before the run cells):
  |delta overall (real - off)| <= 0.65  ->  image channel neutral on ViProKG at lambda 8
  group deltas: reported, with CIs, no thresholds


In [2]:
import csv, glob, hashlib, json, os, random, shutil, statistics as st, subprocess, sys, time
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
OUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/mentor_nb4"
SMOKE = bool(os.environ.get("ANH_V2_SMOKE"))   # local test switch only; not used on Kaggle

def run_streaming(cmd, cwd=None, env=None, tail=40):
    """Run a command, print its output line by line, return (returncode, last lines)."""
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = []
    for line in p.stdout:
        print(line, end="", flush=True)
        last.append(line); last = last[-tail:]
    return p.wait(), last

hits = sorted(glob.glob(f"{INPUT}/**/scripts/verify_tables.py", recursive=True))
assert hits, "Attach the release dataset (the folder containing scripts/verify_tables.py)."
REL = os.path.dirname(os.path.dirname(hits[0]))
EXT = f"{REL}/data/experimental/v3/ext/ViMMKG2"
EDGES = f"{REL}/data/experimental/v3/canh_v2/v10_canh_nhac_ten_ViMMKG2_v2.tsv"
assert hashlib.sha256(open(EDGES, "rb").read()).hexdigest()[:16] == "cefc05167b7123d3", "mention-edge file differs from the signed one"
assert "--tag_b" in open(f"{REL}/scripts/score_anh_v2.py").read(), "release dataset too old: attach v1.2.0 or later"
PHOBERT = next((dp for dp, _, fns in sorted(os.walk(INPUT))
                if "config.json" in fns and ("pytorch_model.bin" in fns or "model.safetensors" in fns) and "phobert" in dp.lower()), None)
print("release:", REL, "\nphobert:", PHOBERT)

release: /kaggle/input/datasets/<kaggle-user>/vimmkg2-release-v1-2-1-rc1 
phobert: /kaggle/input/models/hoanghungnguynkhnh/phobert/pytorch/default/1/phobert


In [3]:
W = f"{OUT_ROOT}/mentor"; D = f"{W}/data/ViMMKG2"; RES = f"{W}/result_anh_v2_off"
os.makedirs(D, exist_ok=True); os.makedirs(RES, exist_ok=True)
for f in os.listdir(f"{REL}/src"):
    if f.endswith(".py"): shutil.copy(f"{REL}/src/{f}", f"{W}/{f}")
for f in os.listdir(EXT):
    if f.endswith((".pt", ".txt", ".tsv", ".json")): shutil.copy(f"{EXT}/{f}", f"{D}/{f}")
def sha16(*ps):
    h = hashlib.sha256()
    for p in sorted(ps): h.update(open(p, "rb").read())
    return h.hexdigest()[:16]
import torch
N_ENT = sum(1 for _ in open(f"{D}/entities.txt", encoding="utf-8"))
M = torch.load(f"{D}/V_mask.pt", map_location="cpu")
print("entities", N_ENT, "| with image", int((~M).sum()), "| inputs sha:", sha16(f"{D}/V.pt", f"{D}/V_mask.pt", f"{D}/L_feat.pt"))
if not SMOKE:
    assert N_ENT == 8255 and int((~M).sum()) == 7790, "unexpected ViProKG files"

entities 8255 | with image 7790 | inputs sha: 69b17297aeb4e95e


## 1. Build the text tensor `T_ctx.pt` (PhoBERT mean pooling; released code)

In [4]:
sys.path.insert(0, W)
from build_text_context import build_text_context, load_encoder
if not os.path.exists(f"{D}/T_ctx.pt"):
    assert PHOBERT, "Attach a PhoBERT checkpoint dataset (path containing 'phobert')."
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load_encoder(PHOBERT, device)
    _ids = json.load(open(f"{D}/ViMMKG2-textual-phobert-seg.json"))["0"]
    print("decode check (entity 0, should read as Vietnamese text):", tok.decode([x for x in _ids if x < tok.vocab_size], skip_special_tokens=True)[:150])
    T, _ = build_text_context("ViMMKG2", D, f"{D}/ViMMKG2-textual-phobert-seg.json", tok, model, batch=64, device=device)
    torch.save(T, f"{D}/T_ctx.pt")
    del model
T = torch.load(f"{D}/T_ctx.pt", map_location="cpu")
nz = int((T.abs().sum(1) > 0).sum())
TCTX = sha16(f"{D}/T_ctx.pt")
print("T_ctx", tuple(T.shape), "rows with text:", nz, "| sha16:", TCTX, "(signed run: a9b350f5cf651d7c)")
if not SMOKE:
    assert tuple(T.shape) == (8255, 768) and nz == 8207, "T_ctx build did not give the expected shape / coverage"
    assert TCTX == "a9b350f5cf651d7c", "T_ctx differs from the signed run: wrong PhoBERT checkpoint?"

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /kaggle/input/models/hoanghungnguynkhnh/phobert/pytorch/default/1/phobert
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


decode check (entity 0, should read as Vietnamese text): Làng cổ Long_Tuyền . human settlement . Làng cổ Long_Tuyền là một làng cổ ở Nam_Bộ thuộc phường Long_Tuyền , thành_phố Cần_Thơ , Việt_Nam .
T_ctx (8255, 768) rows with text: 8207 | sha16: a9b350f5cf651d7c (signed run: a9b350f5cf651d7c)


## 2. Train the off branch (3 seeds; same flags as the signed run except `--use_image 0`)

In [5]:
SEEDS = [int(x) for x in os.environ.get("ANH_V2_SEEDS", "42,7,2024").split(",")]
EPOCHS = os.environ.get("MENTOR_EPOCHS", "150"); VALID_EVERY = os.environ.get("MENTOR_VALID_EVERY", "5")
BRANCHES = ["off"] + (["real"] if SMOKE else [])      # real branch comes from the release on Kaggle
BUDGET_MIN = 60.0
def cmd_for(tag, seed):
    return [sys.executable, "-u", "train.py", "--data_dir", D, "--t_path", f"{D}/T_ctx.pt", "--v_path", f"{D}/V.pt",
            "--use_text", "1", "--use_image", "0" if tag == "off" else "1", "--variant", "LSR", "--lsr_L", f"{D}/L_feat.pt",
            "--lsr_off", "RM", "--lsr_l1", "1",
            "--dump_full", f"{RES}/dump_test_{tag}_s{seed}.tsv", "--dump_full_valid", f"{RES}/dump_valid_{tag}_s{seed}.tsv",
            "--num_epoch", EPOCHS, "--valid_epoch", VALID_EVERY, "--lr", "0.0005", "--batch_size", "512", "--dim", "400",
            "--dim_rel", "64", "--reciprocal", "1", "--entity_bias", "0", "--text_mode", "mean", "--gate_mode", "gate_off",
            "--lambda_align", "8.0", "--seed", str(seed), "--exp", f"anh_v2_{tag}_s{seed}", "--scores_g", "0.15"]
def zipit():
    if os.path.isdir("/kaggle/working"):
        subprocess.run(["bash", "-c", f"cd {W} && zip -qrX /kaggle/working/anh_v2_off_out.zip result_anh_v2_off -x '*.pt'"], check=False)
t_all = time.time()
env = dict(os.environ, VIS_DIM="32")
for s in SEEDS:
    for tag in BRANCHES:
        if os.path.exists(f"{RES}/dump_test_{tag}_s{s}.tsv"):
            print(f"[{tag} s{s}] already done, skipped"); continue
        if (time.time() - t_all) / 60 > BUDGET_MIN:
            print("time budget reached; results so far are saved"); zipit(); raise SystemExit
        c = cmd_for(tag, s); t0 = time.time()
        rc, last = run_streaming(c, cwd=W, env=env)
        print(f"\n[{tag} s{s}] rc={rc} {(time.time() - t0) / 60:.1f} min", flush=True)
        assert rc == 0, f"training failed ({tag}, seed {s}); see the output above"
        json.dump({"cmd": c, "seed": s, "tag": tag, "train_py_sha16": sha16(f"{W}/train.py"), "T_ctx_sha16": TCTX,
                   "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"},
                  open(f"{RES}/config_{tag}_s{s}.json", "w"), indent=1)
        zipit()
print(f"all runs done in {(time.time() - t_all) / 60:.1f} min")

[MENTOR] gate=gate_off seed=42 device=cuda lambda1=8.0 lambda2=0.01
[gate] prior = hard-coded TIER_PRIOR (3 cells)
[features] merged V (8255, 1, 32) -> (8255, 32) (masked mean, 7790/8255 entities have images; mask=V_mask.pt)
[MENTOR] RECIPROCAL: rel 81->162, train 29355->58710
[variant] LSR | paper defaults: {'nguon': 'lexical-semantic router v1', 'dim_L': 64, 'dim_R': 32, 'top_k': 100, 'doi': 's = g_S(q)*S + g_L(q)*L + g_M(q)*M; g initialised ~ (1,0,0); per-query z-score'}
[LSR] active blocks: S + L | g FIXED (1, 0.15, 0), NO R
  use_text=True use_image=False | L_align=True L_gate=False
epoch 1	loss 6041.3605	1s
  [gold-invariance] PASS (|delta| tail 0 - head 0, n=256, split=valid)
  [valid e5] MRR 0.4363 H@1 0.3677 H@10 0.5587 *best
  [LSR g e5] S 1.0000 | L 0.1500 | M 0.0000
epoch 10	loss 4884.6673	9s
  [gold-invariance] PASS (|delta| tail 0 - head 0, n=256, split=valid)
  [valid e10] MRR 0.4374 H@1 0.3681 H@10 0.5641 *best
  [LSR g e10] S 1.0000 | L 0.1500 | M 0.0000
  [gold-invari

## 3. Score: paired delta (real - off), CIs and the pre-specified criterion

In [6]:
if not SMOKE:   # real-branch ranks shipped in the release (bit-identical to the signed run)
    for s in SEEDS:
        src = f"{REL}/data/experimental/v3/ranks_v2/anh_v2_real_s{s}/dump_test.tsv"
        assert os.path.exists(src), f"missing {src} in the release dataset"
        shutil.copy(src, f"{RES}/dump_test_real_s{s}.tsv")
rc, _ = run_streaming([sys.executable, f"{REL}/scripts/score_anh_v2.py", RES, "--tag_b", "off", "--ext", D, "--edges", EDGES,
                       "--out_json", f"{RES}/score_anh_v2_off.json"])
assert rc == 0
res = json.load(open(f"{RES}/score_anh_v2_off.json"))
print("\nVERDICT (pre-specified |delta overall (real - off)| <= 0.65):", "PASS" if res["criterion"]["pass"] else "FAIL",
      f"(delta overall = {res['criterion']['delta_overall']:+.3f})")
zipit()

group                    n    real     off    delta     CI seeds (t,df=2)      CI key bootstrap
overall|all           5274   41.20   41.11   +0.097  [-0.857; +1.051]  [-0.181; +0.372]
overall|overlap       1364   50.59   50.54   +0.042  [-1.008; +1.093]  [-0.406; +0.539]
overall|free          3910   37.93   37.82   +0.116  [-0.829; +1.061]  [-0.215; +0.455]
ordinary|all          3300   45.58   45.63   -0.049  [-0.802; +0.704]  [-0.350; +0.249]
ordinary|overlap      1348   50.98   50.93   +0.043  [-1.051; +1.137]  [-0.452; +0.498]
ordinary|free         1952   41.85   41.97   -0.112  [-0.898; +0.673]  [-0.539; +0.284]
slice|all             1974   33.89   33.55   +0.341  [-1.057; +1.739]  [-0.137; +0.848]
slice|overlap           16   17.76   17.78   -0.018  [-2.700; +2.664]  [-1.006; +0.653]
slice|free            1958   34.02   33.68   +0.344  [-1.087; +1.775]  [-0.150; +0.810]

CRITERION |delta overall| <= 0.65: delta = +0.097 -> PASS
informational (no gate): real branch 41.20 vs signed 